In [1]:
import cv2
import pandas as pd
import pytesseract
import os
import subprocess
import glob
import numpy as np

In [ ]:
# Path to the Tesseract executable 
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'
video_path = 'Streams/HLE vs T1 Game1.mp4'
cap = cv2.VideoCapture(video_path)
fps = round(cap.get(cv2.CAP_PROP_FPS)) 

extracted_data = []
frame_count = 0

# Tesseract configurations
time_config_colon = r'--oem 3 --psm 7 -c tessedit_char_whitelist=0123456789:'
time_config_digits = r'--oem 3 --psm 7 -c tessedit_char_whitelist=0123456789'
gold_config = r'--oem 3 --psm 7 -c tessedit_char_whitelist=0123456789.Kk'
tower_config = r'--oem 3 --psm 8 -c tessedit_char_whitelist=0123456789Oo'

prev_time = ""

def extract_time_robust(time_crop):
    global prev_time
    import re
    
    # First attempt: OCR with colon
    raw = pytesseract.image_to_string(time_crop, config=time_config_colon).strip()
    if re.match(r'^\d{2}:\d{2}$', raw):
        mm, ss = int(raw[:2]), int(raw[3:])
        if ss < 60 and mm < 60:
            prev_time = raw
            return raw
    
    # Only digits, no colon
    raw = pytesseract.image_to_string(time_crop, config=time_config_digits).strip()
    digits = re.sub(r'[^0-9]', '', raw)
    
    result = ""
    if len(digits) == 4:
        result = f"{digits[:2]}:{digits[2:]}"
    elif len(digits) == 5:  
        result = f"{digits[:2]}:{digits[3:]}"
    elif len(digits) == 3:
        result = f"0{digits[0]}:{digits[1:]}"
    
    if result:
        mm, ss = int(result[:2]), int(result[3:])
        if ss < 60 and mm < 60:
            prev_time = result
            return result
    
    # Nothing valid found, return previous time
    return prev_time


while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    if frame_count % fps == 0:
        if frame is None:
            continue
            
        # Cropping the top HUD area from the frame
        y_start, y_end = 0, 150
        x_start, x_end = 380, 1540
        top_hud = frame[y_start:y_end, x_start:x_end]
        
        # Gray-scaling and thresholding to create a binary image for better OCR performance
        gray_hud = cv2.cvtColor(top_hud, cv2.COLOR_BGR2GRAY)
        _, binary_hud = cv2.threshold(gray_hud, 150, 255, cv2.THRESH_BINARY_INV)
        
        # Cropping smaller ROIs (Region of Interest) from the binary HUD
        
        # Time
        time_crop = binary_hud[55:120, 520:640] 
        
        # Blue team gold 
        blue_gold_crop = binary_hud[15:75, 350:450]
        
        # Red team gold 
        red_gold_crop = binary_hud[15:75, 700:800]

        # Blue team towers
        blue_tower_crop = binary_hud[20:60, 310:350]

        # Red team towers
        red_tower_crop = binary_hud[20:60, 810:850]
                
        
        # HUD visibility check (if there are not enough white pixels, the HUD is not visible)
        if cv2.countNonZero(cv2.bitwise_not(time_crop)) < 50 or cv2.countNonZero(cv2.bitwise_not(blue_gold_crop)) < 50 or cv2.countNonZero(cv2.bitwise_not(red_gold_crop)) < 50:
            time_str = prev_time  # HUD not visible, previous time 
            blue_gold_str = ""
            red_gold_str = ""
            blue_tower_str = ""
            red_tower_str = ""
        else:
            # OCR processing on the cropped images
            # OCR processing (szoveg csere O -> 0)
            time_str = extract_time_robust(time_crop)
            blue_gold_str = pytesseract.image_to_string(blue_gold_crop, config=gold_config).strip().upper()
            red_gold_str = pytesseract.image_to_string(red_gold_crop, config=gold_config).strip().upper()
            blue_tower_str = pytesseract.image_to_string(blue_tower_crop, config=tower_config).strip().replace('O', '0').replace('o', '0')
            red_tower_str = pytesseract.image_to_string(red_tower_crop, config=tower_config).strip().replace('O', '0').replace('o', '0')
        
        # Visualization of the cropped areas for debugging purposes
        cv2.imshow('Time', time_crop)
        #cv2.imshow('Blue Gold', blue_gold_crop)
        #cv2.imshow('Red Gold', red_gold_crop)
        #cv2.imshow('Blue Towers', blue_tower_crop)
        #cv2.imshow('Red Towers', red_tower_crop)
        
        print(f"[{frame_count//fps}s] Time: {time_str} | Blue Gold: {blue_gold_str} | Red Gold: {red_gold_str} | Blue Towers: {blue_tower_str} | Red Towers: {red_tower_str}")
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            print("Processing interrupted.")
            break
        
        # Saving the extracted data into a list of dictionaries
        extracted_data.append({
            'video_sec': frame_count // fps,
            'match_time': time_str,
            'blue_total_gold': blue_gold_str,
            'red_total_gold': red_gold_str,
            'blue_towers': blue_tower_str,
            'red_towers': red_tower_str
        })
        
    frame_count += 1

cap.release()
cv2.destroyAllWindows()

df = pd.DataFrame(extracted_data)
print(df.head())
df.to_csv('extracted_data.csv', index=False)

[0s] Time:  | Blue Gold:  | Red Gold:  | Blue Towers:  | Red Towers: 
[1s] Time:  | Blue Gold:  | Red Gold:  | Blue Towers:  | Red Towers: 
[2s] Time:  | Blue Gold:  | Red Gold:  | Blue Towers:  | Red Towers: 
[3s] Time:  | Blue Gold:  | Red Gold:  | Blue Towers:  | Red Towers: 
[4s] Time:  | Blue Gold:  | Red Gold:  | Blue Towers:  | Red Towers: 
[5s] Time:  | Blue Gold:  | Red Gold:  | Blue Towers:  | Red Towers: 
[6s] Time:  | Blue Gold:  | Red Gold:  | Blue Towers:  | Red Towers: 
[7s] Time:  | Blue Gold:  | Red Gold:  | Blue Towers:  | Red Towers: 
[8s] Time:  | Blue Gold:  | Red Gold:  | Blue Towers:  | Red Towers: 
[9s] Time:  | Blue Gold:  | Red Gold:  | Blue Towers:  | Red Towers: 
[10s] Time:  | Blue Gold:  | Red Gold:  | Blue Towers:  | Red Towers: 
[11s] Time:  | Blue Gold:  | Red Gold:  | Blue Towers:  | Red Towers: 
[12s] Time:  | Blue Gold:  | Red Gold:  | Blue Towers:  | Red Towers: 
[13s] Time:  | Blue Gold:  | Red Gold:  | Blue Towers:  | Red Towers: 
[14s] Time:  | B

In [30]:
# Processing the lower HUD area for KDA, CS and Items extraction (WITH VISUAL DEBUG)
video_path = 'Streams/HLE vs T1 Game1.mp4'
if not os.path.exists(video_path):
    video_path = 'e:/BME/League-of-Legends-Esport-Analitika-2026/Streams/HLE vs T1 Game1.mp4'

items_dir = 'Items'
if not os.path.exists(items_dir):
    items_dir = 'e:/BME/League-of-Legends-Esport-Analitika-2026/Items'

cap = cv2.VideoCapture(video_path)
fps = round(cap.get(cv2.CAP_PROP_FPS))
if fps <= 0: fps = 30

# --- ITEM RECOGNITION SETUP ---
print("Tárgyak betöltése...")
item_size = 22
item_files = glob.glob(os.path.join(items_dir, '*.png'))
items = {}

for f in item_files:
    item_id = os.path.basename(f).split('.')[0]
    img = cv2.imread(f)
    if img is not None:
        img_resized = cv2.resize(img, (item_size, item_size), interpolation=cv2.INTER_AREA)
        items[item_id] = cv2.cvtColor(img_resized, cv2.COLOR_BGR2GRAY)

print(f"{len(items)} tárgy betöltve.")

def identify_item(crop, items_dict):
    core = crop[2:16, 2:20]
    if np.mean(core) < 25:
        return "Empty"
        
    best_id = "Empty"
    best_score = -1
    crop_gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    
    for item_id, item_gray in items_dict.items():
        result = cv2.matchTemplate(crop_gray, item_gray, cv2.TM_CCOEFF_NORMED)
        score = result[0][0]
        if score > best_score:
            best_score = score
            best_id = item_id
            
    return best_id if best_score > 0.45 else "Empty"

y_offsets = [13, 52, 90, 129, 167]
b_x_trinket = 16
b_x_items = [94, 118, 142, 166, 190, 214]

r_x_items = [0, 24, 48, 72, 96, 120]
r_x_trinket = 190

roles = ["top", "jungle", "mid", "adc", "support"]

# --- OCR SETUP ---
extracted_lower_data = []
frame_count = 0

custom_config_kda = r'--oem 3 --psm 6 -c tessedit_char_whitelist=0123456789/'
custom_config_cs = r'--oem 3 --psm 6 -c tessedit_char_whitelist=0123456789'

def get_binary(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    scaled = cv2.resize(gray, None, fx=2.5, fy=2.5, interpolation=cv2.INTER_CUBIC)
    _, binary = cv2.threshold(scaled, 160, 255, cv2.THRESH_BINARY_INV)
    return binary

def do_ocr(binary_image, config):
    return pytesseract.image_to_string(binary_image, config=config).strip()

print("Alsó HUD feldolgozás indítása...")
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    if frame_count % fps == 0:
        if frame is None:
            continue
            
        height, width = frame.shape[:2]
        y_start = int(height * 0.75)
        y_end = int(height)
        
        blue_kda_crop = frame[y_start:y_end, 790:875]
        blue_cs_crop  = frame[y_start:y_end, 860:900]
        red_cs_crop   = frame[y_start:y_end, 1020:1060]
        red_kda_crop  = frame[y_start:y_end, 1045:1130]
        
        b_kda_bin = get_binary(blue_kda_crop)
        b_cs_bin  = get_binary(blue_cs_crop)
        r_cs_bin  = get_binary(red_cs_crop)
        r_kda_bin = get_binary(red_kda_crop)
        
        lower_hud = frame[y_start:y_end, 500:1400]
        
        # 1. Definiáljuk a kezdőpontokat a lower_hud-hoz képest
        blue_offset_y, blue_offset_x = 25, 40
        red_offset_y, red_offset_x = 40, 650
        
        # 2. A kivágások a lower_hud-ból (a magasság végéig megyünk)
        blue_inv = lower_hud[blue_offset_y:, blue_offset_x:260]
        red_inv = lower_hud[red_offset_y:, red_offset_x:880]
        
        debug_hud = lower_hud.copy()
        
        current_data = {'video_sec': frame_count // fps}
        
        if cv2.mean(blue_kda_crop)[0] > 100:  
            current_data['blue_kda'], current_data['blue_cs'], current_data['red_cs'], current_data['red_kda'] = "", "", "", ""
            for team in ['blue', 'red']:
                for role in roles:
                    for i in range(1, 7): current_data[f'{team}_{role}_item{i}'] = ""
                    current_data[f'{team}_{role}_trinket'] = ""
            
            cv2.putText(debug_hud, "HUD NOT VISIBLE", (400, 100), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
            
        else:
            current_data['blue_kda'] = " ".join(do_ocr(b_kda_bin, custom_config_kda).split())
            current_data['blue_cs'] = " ".join(do_ocr(b_cs_bin, custom_config_cs).split())
            current_data['red_cs'] = " ".join(do_ocr(r_cs_bin, custom_config_cs).split())
            current_data['red_kda'] = " ".join(do_ocr(r_kda_bin, custom_config_kda).split())
            
            print(f"\n--- [{current_data['video_sec']}s] ADATOK ---")
            print(f"B-KDA: {current_data['blue_kda']} | B-CS: {current_data['blue_cs']} | R-CS: {current_data['red_cs']} | R-KDA: {current_data['red_kda']}")
            
            # ITEMS (Blue)
            for i, y in enumerate(y_offsets):
                role = roles[i]
                found_items = []
                for idx, x in enumerate(b_x_items):
                    crop = blue_inv[y:y+item_size, x:x+item_size]
                    item_id = identify_item(crop, items)
                    current_data[f'blue_{role}_item{idx+1}'] = item_id
                    
                    # 3. Kiszámoljuk a pontos rajzolási koordinátát
                    draw_x = blue_offset_x + x
                    draw_y = blue_offset_y + y
                    
                    if item_id != "Empty":
                        found_items.append(item_id)
                        cv2.rectangle(debug_hud, (draw_x, draw_y), (draw_x+item_size, draw_y+item_size), (0, 255, 0), 1)
                        cv2.putText(debug_hud, item_id, (draw_x, draw_y-2), cv2.FONT_HERSHEY_SIMPLEX, 0.3, (0, 255, 0), 1)
                    else:
                        cv2.rectangle(debug_hud, (draw_x, draw_y), (draw_x+item_size, draw_y+item_size), (0, 0, 255), 1)
                
                trinket_crop = blue_inv[y:y+item_size, b_x_trinket:b_x_trinket+item_size]
                tr_id = identify_item(trinket_crop, items)
                current_data[f'blue_{role}_trinket'] = tr_id
                
                draw_x_tr = blue_offset_x + b_x_trinket
                if tr_id != "Empty":
                    found_items.append(f"T:{tr_id}")
                    cv2.rectangle(debug_hud, (draw_x_tr, draw_y), (draw_x_tr+item_size, draw_y+item_size), (0, 255, 255), 1)
                else:
                    cv2.rectangle(debug_hud, (draw_x_tr, draw_y), (draw_x_tr+item_size, draw_y+item_size), (0, 0, 255), 1)
                
                if i in [0, 3]:  # Csak a Top-ot és ADC-t printeljük ki
                    print(f"  Blue {role.capitalize()} Items: {found_items}")
            
            # ITEMS (Red)
            for i, y in enumerate(y_offsets):
                role = roles[i]
                found_items = []
                for idx, x in enumerate(r_x_items):
                    crop = red_inv[y:y+item_size, x:x+item_size]
                    item_id = identify_item(crop, items)
                    current_data[f'red_{role}_item{idx+1}'] = item_id
                    
                    # 4. Kiszámoljuk a pontos rajzolási koordinátát a piros csapatnál is
                    draw_x = red_offset_x + x
                    draw_y = red_offset_y + y
                    
                    if item_id != "Empty":
                        found_items.append(item_id)
                        cv2.rectangle(debug_hud, (draw_x, draw_y), (draw_x+item_size, draw_y+item_size), (0, 255, 0), 1)
                    else:
                        cv2.rectangle(debug_hud, (draw_x, draw_y), (draw_x+item_size, draw_y+item_size), (0, 0, 255), 1)
                
                trinket_crop = red_inv[y:y+item_size, r_x_trinket:r_x_trinket+item_size]
                tr_id = identify_item(trinket_crop, items)
                current_data[f'red_{role}_trinket'] = tr_id
                
                draw_x_tr = red_offset_x + r_x_trinket
                if tr_id != "Empty":
                    found_items.append(f"T:{tr_id}")
                    cv2.rectangle(debug_hud, (draw_x_tr, draw_y), (draw_x_tr+item_size, draw_y+item_size), (0, 255, 255), 1)
                else:
                    cv2.rectangle(debug_hud, (draw_x_tr, draw_y), (draw_x_tr+item_size, draw_y+item_size), (0, 0, 255), 1)
                    
                if i in [0, 3]:
                    print(f"  Red {role.capitalize()} Items: {found_items}")
        
        extracted_lower_data.append(current_data)
        
        cv2.imshow('Lower HUD - Item Detection', debug_hud)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            print("Feldolgozás megszakítva.")
            break
            
    frame_count += 1

cap.release()
cv2.destroyAllWindows()

df_lower = pd.DataFrame(extracted_lower_data)
output_path = 'hle_vs_t1_game1_lower_data.csv'
df_lower.to_csv(output_path, index=False)
print(f"\nKész! Adatok elmentve: {output_path}")

Tárgyak betöltése...
960 tárgy betöltve.
Alsó HUD feldolgozás indítása...

--- [0s] ADATOK ---
B-KDA:  | B-CS:  | R-CS:  | R-KDA: 
  Blue Top Items: ['223026', '3144', '226699', '223002', 'T:224005']
  Blue Adc Items: []
  Red Top Items: ['1211', '1510', '226699', '2144', '5003', '1211', 'T:224644']
  Red Adc Items: []

--- [1s] ADATOK ---
B-KDA:  | B-CS:  | R-CS:  | R-KDA: 
  Blue Top Items: ['771004', '226632', '1029', 'T:3044']
  Blue Adc Items: ['1510']
  Red Top Items: []
  Red Adc Items: []

--- [2s] ADATOK ---
B-KDA:  | B-CS:  | R-CS:  | R-KDA: 
  Blue Top Items: ['226656', '773065', '3144']
  Blue Adc Items: ['3144', '226632', '3144']
  Red Top Items: ['223026', '773071', '223026']
  Red Adc Items: []

--- [3s] ADATOK ---
B-KDA:  | B-CS:  | R-CS:  | R-KDA: 
  Blue Top Items: ['3144', '226333', '1001', 'T:773065']
  Blue Adc Items: []
  Red Top Items: []
  Red Adc Items: []

--- [4s] ADATOK ---
B-KDA:  | B-CS:  | R-CS:  | R-KDA: 
  Blue Top Items: ['226632', '773141', '773065', 

In [3]:
video_path = 'Streams/HLE vs T1 Game2.mp4'
if not os.path.exists(video_path):
    video_path = 'e:/BME/League-of-Legends-Esport-Analitika-2026/Streams/HLE vs T1 Game1.mp4'

TILES_DIR = 'Champion tiles'
if not os.path.exists(TILES_DIR):
    TILES_DIR = 'e:/BME/League-of-Legends-Esport-Analitika-2026/Champion tiles'

cap = cv2.VideoCapture(video_path)
fps = round(cap.get(cv2.CAP_PROP_FPS))
if fps <= 0: fps = 30
print(f"FPS: {fps}")

GAME_START_OFFSET = 90
MAX_SECONDS = 600 

# Prepare SIFT feature extractor
print("Creating SIFT feature extractor...")
sift = cv2.SIFT_create()

base_names = set()
if os.path.exists(TILES_DIR):
    for f in os.listdir(TILES_DIR):
        if f.endswith('_0.jpg') and f.count('_') == 1:
            base_names.add(f.replace('_0.jpg', ''))

def get_base_name(filename):
    name_no_ext = filename.rsplit('.', 1)[0]
    parts = name_no_ext.rsplit('_', 1)
    if len(parts) == 2:
        full_name = parts[0]
        if full_name in base_names: return full_name
        sub_parts = full_name.split('_')
        for i in range(len(sub_parts)):
            candidate = '_'.join(sub_parts[i:])
            if candidate in base_names: return candidate
    return None

grouped_sift = {}
if os.path.exists(TILES_DIR):
    for f in os.listdir(TILES_DIR):
        if f.endswith('.jpg'):
            base = get_base_name(f)
            if not base: continue
            img = cv2.imread(os.path.join(TILES_DIR, f))
            if img is not None:
                h, w = img.shape[:2]
                margin_x = int(w * 0.15)
                face = img[0:int(h*0.65), margin_x:w-margin_x]
                gray = cv2.cvtColor(face, cv2.COLOR_BGR2GRAY)
                kp, des = sift.detectAndCompute(gray, None)
                if des is not None:
                    if base not in grouped_sift: grouped_sift[base] = []
                    grouped_sift[base].append(des)
print(f"{len(grouped_sift)} basic champions loaded with their skins.")

# Multi-frame scanning for champion identification
print("\nMulti-frame scanning for champion identification (30s, 60s, 90s, 120s, 150s)...")
bf = cv2.BFMatcher()

portrait_coords = {
    'blue_top': (140, 190, 65, 115), 'blue_jungle': (255, 305, 65, 115),
    'blue_mid': (370, 420, 65, 115), 'blue_adc': (470, 520, 65, 115), 'blue_support': (585, 635, 65, 115),
    'red_top': (140, 190, 1805, 1855), 'red_jungle': (255, 305, 1805, 1855),
    'red_mid': (370, 420, 1805, 1855), 'red_adc': (470, 520, 1805, 1855), 'red_support': (585, 635, 1805, 1855)
}

accum_scores = {role: {} for role in portrait_coords}
check_frames = [30, 60, 90, 120, 150]

for t in check_frames:
    cap.set(cv2.CAP_PROP_POS_FRAMES, (GAME_START_OFFSET + t) * fps)
    ret, frame = cap.read()
    if not ret: continue
    
    for role, (y1, y2, x1, x2) in portrait_coords.items():
        crop = frame[y1:y2, x1:x2]
        roi = crop[0:45, 0:50]
        upscaled = cv2.resize(roi, (roi.shape[1]*3, roi.shape[0]*3), interpolation=cv2.INTER_CUBIC)
        gray = cv2.cvtColor(upscaled, cv2.COLOR_BGR2GRAY)
        kp, des = sift.detectAndCompute(gray, None)
        
        if des is not None and len(des) > 1:
            for name, des_list in grouped_sift.items():
                champ_best = 0
                for ref_des in des_list:
                    if ref_des is not None and len(ref_des) > 1:
                        matches = bf.knnMatch(des, ref_des, k=2)
                        good = sum(1 for m_list in matches if len(m_list) == 2 and m_list[0].distance < 0.75 * m_list[1].distance)
                        champ_best = max(champ_best, good)
                accum_scores[role][name] = accum_scores[role].get(name, 0) + champ_best

current_teams = {}
print("\nChampion identification results:")
for role in portrait_coords:
    if accum_scores[role]:
        best_champ = max(accum_scores[role].items(), key=lambda x: x[1])[0]
    else:
        best_champ = "Unknown"
    current_teams[role] = best_champ
    print(f"  {role}: {best_champ}")

# Main video processing for champion data extraction
extracted_data = []
processed_seconds = 0
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

print("Starting per-second data extraction...")
while True:
    target_frame = int(processed_seconds * fps)
    cap.set(cv2.CAP_PROP_POS_FRAMES, target_frame)
    ret, frame = cap.read()

    if not ret or (MAX_SECONDS is not None and processed_seconds >= MAX_SECONDS):
        break

    match_seconds = max(0, processed_seconds - GAME_START_OFFSET)
    minutes = match_seconds // 60
    seconds = match_seconds % 60
    match_time = f"{minutes:02d}:{seconds:02d}"

    if processed_seconds % 30 == 0 and processed_seconds > 5:
        print(f"[{match_time}] Processing...")

    extracted_data.append({
        'video_sec': processed_seconds,
        'match_time': match_time,
        'blue_top_champ': current_teams['blue_top'],
        'blue_jungle_champ': current_teams['blue_jungle'],
        'blue_mid_champ': current_teams['blue_mid'],
        'blue_adc_champ': current_teams['blue_adc'],
        'blue_support_champ': current_teams['blue_support'],
        'red_top_champ': current_teams['red_top'],
        'red_jungle_champ': current_teams['red_jungle'],
        'red_mid_champ': current_teams['red_mid'],
        'red_adc_champ': current_teams['red_adc'],
        'red_support_champ': current_teams['red_support']
    })

    processed_seconds += 1

cap.release()

df = pd.DataFrame(extracted_data)
output_path = 'extracted_champions.csv'
df.to_csv(output_path, index=False)

print(f"\nProcessing complete! Total: {len(df)} seconds.")
print(f"Data saved to: {output_path}")


FPS: 60
Creating SIFT feature extractor...
173 basic champions loaded with their skins.

Multi-frame scanning for champion identification (30s, 60s, 90s, 120s, 150s)...

Champion identification results:
  blue_top: Graves
  blue_jungle: Pantheon
  blue_mid: Syndra
  blue_adc: Yunara
  blue_support: Lulu
  red_top: Galio
  red_jungle: JarvanIV
  red_mid: Yone
  red_adc: Caitlyn
  red_support: Karma
Starting per-second data extraction...
[00:00] Processing...
[00:00] Processing...
[00:00] Processing...
[00:30] Processing...
[01:00] Processing...
[01:30] Processing...
[02:00] Processing...
[02:30] Processing...
[03:00] Processing...
[03:30] Processing...
[04:00] Processing...
[04:30] Processing...
[05:00] Processing...
[05:30] Processing...
[06:00] Processing...
[06:30] Processing...
[07:00] Processing...
[07:30] Processing...
[08:00] Processing...

Processing complete! Total: 600 seconds.
Data saved to: extracted_champions.csv


In [2]:
def split_video_ffmpeg(input_video, output_dir, segment_length=30):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # Setup the output pattern for the segmented clips
    output_pattern = os.path.join(output_dir, "clip_%03d.mp4")

    # FFmpeg
    command = [
        'ffmpeg',
        '-i', input_video,
        '-c', 'copy',                 
        '-map', '0',                  # Keep all streams (audio/video)
        '-segment_time', str(segment_length),
        '-f', 'segment',              # Segmentation enabled
        '-reset_timestamps', '1',     # Reset timestamps for each clip
        output_pattern
    ]

    print(f"[{input_video}] Cropping into {segment_length} second segments...")
    
    try:
        subprocess.run(command, check=True)
        print(f"Done! Clips are available in the '{output_dir}' directory.")
    except FileNotFoundError:
        print("Error: FFmpeg is not installed or not in the PATH!")
    except subprocess.CalledProcessError as e:
        print(f"Error occurred during cutting: {e}")

video_file = 'Streams/HLE vs T1 Game1.mp4'
split_video_ffmpeg(video_file, 'CVAT_Clips', segment_length=30)

[Streams/HLE vs T1 Game1.mp4] Cropping into 30 second segments...
Done! Clips are available in the 'CVAT_Clips' directory.


In [13]:
from ultralytics import YOLO

# Egy előre betanított kis modell (nano) betöltése, ami szupergyors és kevés VRAM-ot eszik
model = YOLO('yolov8n.pt') 

# Tanítás indítása (a data paraméterhez a CVAT-ból exportált yaml fájl pontos útvonalát add meg)
# Az epochs=50 elég arra, hogy lásd, tanul-e egyáltalán a háló
results = model.train(data='YOLO_Dataset/data.yaml', epochs=50, imgsz=640)

Ultralytics 8.4.174  Python-3.13.13 torch-2.14.1+cpu CPU (Intel Core i7-8650U 1.90GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=YOLO_Dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-11, nbs=64, nms=None, opset=None, optimize=False, optimize

KeyboardInterrupt: 